In [1]:
import pandas as pd
from scipy. stats import ranksums
from statsmodels.stats.multitest import multipletests
import numpy as np
import os

def perform_differential_analysis(signature_scores, responders, progressors, cell_types):
    """
    Perform differential analysis:  Mean(Responders) - Mean(Progressors)
    
    Returns
    -------
    pd.DataFrame with columns:  Cell Type, P_values, Q_values, Diff_Mean
    """
    pvalues = []
    diff_means = []
    
    for cell_type in cell_types: 
        # Get scores
        responders_score = signature_scores. loc[responders, cell_type]. values
        progressors_score = signature_scores.loc[progressors, cell_type].values
        
        # Remove NaN
        responders_score = responders_score[~np.isnan(responders_score)]
        progressors_score = progressors_score[~np.isnan(progressors_score)]
        
        # Calculate Diff_Mean = Mean(Responders) - Mean(Progressors)
        diff_mean = np.mean(responders_score) - np.mean(progressors_score)
        diff_means.append(diff_mean)
        
        # Statistical test
        if len(responders_score) > 0 and len(progressors_score) > 0:
            _, pvalue = ranksums(responders_score, progressors_score)
            pvalues.append(pvalue)
        else:
            pvalues. append(np.nan)
    
    # FDR correction
    valid_pvalues = [p for p in pvalues if not np.isnan(p)]
    if len(valid_pvalues) > 0:
        _, pvals_corrected_valid, _, _ = multipletests(valid_pvalues, alpha=0.05, method='fdr_bh')
        
        # Map back to original list
        pvals_corrected = []
        valid_idx = 0
        for p in pvalues:
            if np.isnan(p):
                pvals_corrected.append(np. nan)
            else:
                pvals_corrected.append(pvals_corrected_valid[valid_idx])
                valid_idx += 1
    else:
        pvals_corrected = [np.nan] * len(pvalues)
    
    # Create result dataframe
    result_dict = {
        'Cell Type': cell_types,
        'P_values': pvalues,
        'Q_values': pvals_corrected,
        'Diff_Mean': diff_means  # Mean(Responders) - Mean(Progressors)
    }
    
    return pd.DataFrame(result_dict)


def main():
    """Main analysis"""
    seeds = [42, 0, 1, 2, 3]
    
    for seed in seeds:
        print(f"\n{'='*60}")
        print(f"Seed: {seed}")
        print(f"{'='*60}")
        
        # Create output directory
        output_dir = f'ResultsDA_cibersortx/Seed_{seed}'
        os.makedirs(output_dir, exist_ok=True)
        
        datasets = [
            f'avatarsk5_{seed}', 
            f'avatarsk10_{seed}',
            f'ctgan_{seed}',
            f'gaussiancopula_{seed}', 
            f'synthpop_{seed}', 
            f'tvae_{seed}'
        ]
        
        for dataset in datasets:
            print(f'\n---{dataset}---')
            
            try:
                # Load CIBERSORTx results
                or_res = pd.read_csv(f"ResultsCibersortx/CIBERSORTx_{dataset}_Results.csv")
                or_cellproportion = or_res.iloc[: , 1:23]
                cell_types = or_cellproportion.columns.tolist()
                
                # Normalize to 100% per sample
                norm_or_cellproportion = or_cellproportion.copy()
                for i in range(or_cellproportion.shape[0]):
                    row_sum = np.sum(norm_or_cellproportion.iloc[i, :]. values)
                    if row_sum > 0:
                        norm_or_cellproportion.iloc[i, : ] = (
                            norm_or_cellproportion.iloc[i, :].values / row_sum * 100
                        )
                
                signature_scores = pd.concat(
                    [pd.DataFrame(or_res["Mixture"]), norm_or_cellproportion],
                    axis=1
                ).set_index('Mixture')
                
                # Load original data
                original_data = pd.read_csv(f'../../Data/{dataset}.csv')
                
                # Add labels
                responder_crit = original_data['ImmunoPhenotype'].isin(['Infiltrated'])
                progressor_crit = original_data['ImmunoPhenotype'].isin(['Desert', 'Excluded'])
                
                original_data['Labels'] = 'Unknown'
                original_data. loc[responder_crit, 'Labels'] = 'Responder'
                original_data.loc[progressor_crit, 'Labels'] = 'Progressor'
                
                # ========================================
                # 1. OVERALL
                # ========================================
                responders = original_data[
                    original_data['Labels'] == 'Responder'
                ]['Patient_ID'].values.tolist()
                
                progressors = original_data[
                    original_data['Labels'] == 'Progressor'
                ]['Patient_ID'].values.tolist()
                
                df_res = perform_differential_analysis(
                    signature_scores, responders, progressors, cell_types
                )
                df_res.to_csv(f'{output_dir}/{dataset}.csv', index=False)
                print(f'  Overall: {len(responders)} responders, {len(progressors)} progressors')
                
            except Exception as e:
                print(f'  ❌ Error:  {str(e)}')
                continue
    
    print(f"\n{'='*60}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()


Seed: 42

---avatarsk5_42---
  Overall: 60 responders, 17 progressors

---avatarsk10_42---
  Overall: 70 responders, 16 progressors

---ctgan_42---
  Overall: 72 responders, 40 progressors

---gaussiancopula_42---
  Overall: 89 responders, 16 progressors

---synthpop_42---
  Overall: 102 responders, 11 progressors

---tvae_42---


/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site

  Overall: 0 responders, 0 progressors

Seed: 0

---avatarsk5_0---
  Overall: 70 responders, 15 progressors

---avatarsk10_0---
  Overall: 76 responders, 14 progressors

---ctgan_0---
  Overall: 79 responders, 25 progressors

---gaussiancopula_0---
  Overall: 86 responders, 26 progressors

---synthpop_0---
  Overall: 85 responders, 23 progressors

---tvae_0---


/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site

  Overall: 0 responders, 0 progressors

Seed: 1

---avatarsk5_1---
  Overall: 62 responders, 12 progressors

---avatarsk10_1---
  Overall: 68 responders, 19 progressors

---ctgan_1---
  Overall: 77 responders, 45 progressors

---gaussiancopula_1---
  Overall: 86 responders, 19 progressors

---synthpop_1---
  Overall: 81 responders, 15 progressors

---tvae_1---


/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site

  Overall: 0 responders, 1 progressors

Seed: 2

---avatarsk5_2---
  Overall: 69 responders, 18 progressors

---avatarsk10_2---
  Overall: 79 responders, 14 progressors

---ctgan_2---
  Overall: 68 responders, 40 progressors

---gaussiancopula_2---
  Overall: 70 responders, 28 progressors

---synthpop_2---
  Overall: 81 responders, 15 progressors

---tvae_2---


/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site

  Overall: 0 responders, 0 progressors

Seed: 3

---avatarsk5_3---
  Overall: 61 responders, 13 progressors

---avatarsk10_3---
  Overall: 72 responders, 9 progressors

---ctgan_3---
  Overall: 65 responders, 35 progressors

---gaussiancopula_3---
  Overall: 83 responders, 23 progressors

---synthpop_3---
  Overall: 81 responders, 9 progressors

---tvae_3---
  Overall: 3 responders, 0 progressors

✅ DONE!


/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site